In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

ROOT = Path.cwd()
DATA_PATH = ROOT / "outputs" / "cleaned" / "songs_pre_imputation_with_missingness_flags.parquet"
TARGET = "mode"
TARGET_TYPE = "classification"

TARGET_COLUMNS = ['valence', 'loudness', 'danceability', 'energy', 'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'key', 'mode']
REGRESSION_TARGETS = ['valence', 'loudness', 'danceability', 'energy', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']
FEATURE_POOL = ['valence', 'loudness', 'danceability', 'energy', 'tempo', 'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'key', 'mode', 'valence_missing', 'loudness_missing', 'danceability_missing', 'energy_missing', 'tempo_missing', 'acousticness_missing', 'instrumentalness_missing', 'liveness_missing', 'speechiness_missing', 'key_missing', 'mode_missing']

feature_columns = [column for column in FEATURE_POOL if column != TARGET]
best_params = {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2.403950683025824, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469887, 'gamma': 0.002570603566117596, 'objective': 'reg:squarederror', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1}


In [ ]:
columns_to_load = sorted(set(FEATURE_POOL + ["track_id", "artist_name", "track_name"]))
df = pd.read_parquet(DATA_PATH, columns=columns_to_load)
df.shape


In [ ]:
rows = len(df)
missing_count = int(df[TARGET].isna().sum())
observed_count = int(df[TARGET].notna().sum())
missing_pct = missing_count / rows
print({"target": TARGET, "type": TARGET_TYPE, "rows": rows, "missing_count": missing_count, "missing_pct": missing_pct, "observed_count": observed_count})


In [ ]:
if TARGET_TYPE == 'regression':
    display(df[TARGET].describe())
else:
    display(df[TARGET].value_counts(dropna=False).sort_index())


In [ ]:
observed_df = df[df[TARGET].notna()].copy()
missing_df = df[df[TARGET].isna()].copy()
X_missing = missing_df[feature_columns]

print({"observed_rows": len(observed_df), "missing_rows": len(missing_df), "feature_count": len(feature_columns)})


In [ ]:
missing_feature_summary = pd.DataFrame({
    "missing_rows_non_null_count": X_missing.notna().sum(),
    "missing_rows_null_count": X_missing.isna().sum(),
    "missing_rows_nunique": X_missing.nunique(),
}).sort_values(['missing_rows_null_count', 'missing_rows_nunique'], ascending=[False, True])
missing_feature_summary


In [ ]:
row_non_null_counts = X_missing.notna().sum(axis=1)
row_non_null_counts.describe()


In [ ]:
missing_df.isna().sum().sort_values(ascending=False)


In [ ]:
missing_df[["track_id", "artist_name", "track_name", TARGET]].head(20)


In [ ]:
if TARGET in REGRESSION_TARGETS:
    display(df[REGRESSION_TARGETS].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False))
else:
    print('Classification target; skipping numeric correlation ranking.')


In [ ]:
if TARGET_TYPE == 'regression' and missing_count > 0:
    X_observed = observed_df[feature_columns]
    y_observed = observed_df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X_observed,
        y_observed,
        test_size=0.2,
        random_state=42,
    )
    model = XGBRegressor(**best_params)
    model.fit(X_train, y_train)
    test_predictions = np.clip(model.predict(X_test), 0, 1)
    display({
        "mae": mean_absolute_error(y_test, test_predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, test_predictions)),
        "r2": r2_score(y_test, test_predictions),
    })
    final_model = XGBRegressor(**best_params)
    final_model.fit(X_observed, y_observed)
    predictions = np.clip(final_model.predict(X_missing), 0, 1)
    display(pd.Series(test_predictions).describe())
    display(pd.Series(test_predictions).nunique())
    display(pd.Series(predictions).describe())
    display(pd.Series(predictions).nunique())
else:
    print('Skipping regressor fit for this target in the diagnostics notebook.')
